# Server-side tools, batch inference and fine-tuning for OpenAI models on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Three things Bedrock runs for you: tools that the model calls from inside a request,
batch jobs, and fine-tuning jobs. All three are for OpenAI's open-weight gpt-oss models.
The hosted GPT models have one server-side tool, web search, in
[`02-web-search-and-grounding`](02-web-search-and-grounding.ipynb).

| Feature | Endpoint | Models (September 2026) |
|---|---|---|
| Tools behind an MCP connector | `bedrock-mantle`, Responses API, bare `/v1` path | gpt-oss-120b and gpt-oss-20b |
| Batch inference | `bedrock-runtime`, `/openai/v1/batches` | gpt-oss and gpt-oss-safeguard, by their `bedrock-runtime` IDs |
| Reinforcement fine-tuning | `bedrock-mantle` in `us-west-2`, bare `/v1` path | gpt-oss-20b, and Qwen3 32B |

The notebook deploys nothing and starts no job. It shows the request for each feature,
makes the read-only calls, and builds and checks a training file and a reward function
locally.

**Covered:** tools behind an MCP connector, batch inference, and reinforcement
fine-tuning.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK, on the two endpoints the
table names. Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json

from aws_bedrock_token_generator import provide_token
from openai import OpenAI

REGION = "us-east-1"
FT_REGION = "us-west-2"  # the Region for fine-tuning through the OpenAI-compatible APIs

# Batch inference: bedrock-runtime's OpenAI-compatible path. Each key is a short-term
# Bedrock API key for its Region, minted from your AWS credentials.
runtime = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
    timeout=60,
)
# Fine-tuning: bedrock-mantle's bare /v1 path, in us-west-2.
tuning = OpenAI(
    base_url=f"https://bedrock-mantle.{FT_REGION}.api.aws/v1",
    api_key=provide_token(region=FT_REGION),
    timeout=60,
)
served = {m.id for m in tuning.models.list()}
for model in ("openai.gpt-oss-20b", "qwen.qwen3-32b"):
    print(f"{model:20} {'on' if model in served else 'NOT on'} bedrock-mantle in {FT_REGION}")

openai.gpt-oss-20b   on bedrock-mantle in us-west-2
qwen.qwen3-32b       on bedrock-mantle in us-west-2


## 1. Tools behind an MCP connector

A server-side tool is a Lambda function, or an Amazon Bedrock AgentCore Gateway, that
speaks the Model Context Protocol. Name it in `tools` with its ARN as `connector_id`.
Bedrock lists its tools, the model calls them during the request, and the reply carries
`mcp_list_tools` and `mcp_call` items, with no tool loop on your side. Bedrock calls the
Lambda as the caller's IAM identity, and the Lambda needs the same IAM policy as the
application that calls the model. The user guide documents this for gpt-oss-120b and
gpt-oss-20b on `bedrock-mantle`. Deploying a Lambda is outside this notebook, so here is
the request:

```python
client = OpenAI(base_url="https://bedrock-mantle.us-east-1.api.aws/v1", api_key=provide_token(region="us-east-1"))
response = client.responses.create(
    model="openai.gpt-oss-120b",
    tools=[{
        "type": "mcp",
        "server_label": "orders",
        "connector_id": "arn:aws:lambda:us-east-1:123456789012:function:order-status",
        "require_approval": "never",  # must be "never"
    }],
    input="What is the status of order 88213?",
    store=False,
)
calls = [item for item in response.output if item.type == "mcp_call"]
```

And a Lambda that serves one tool, `get_order_status`, over `tools/list` and
`tools/call`:

```python
import json


def lambda_handler(event, context):
    """A minimal MCP server with one tool, get_order_status."""
    request_id = event.get("id")
    if event.get("method") == "tools/list":
        tool = {
            "name": "get_order_status",
            "description": "Look up the status of a customer order.",
            "inputSchema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                            "required": ["order_id"]},
        }
        return {"jsonrpc": "2.0", "id": request_id, "result": {"tools": [tool]}}
    if event.get("method") == "tools/call" and event["params"].get("name") == "get_order_status":
        order_id = event["params"].get("arguments", {}).get("order_id")
        status = {"88213": "shipped"}.get(order_id, "not_found")
        text = json.dumps({"order_id": order_id, "status": status})
        return {"jsonrpc": "2.0", "id": request_id, "result": {"content": [{"type": "text", "text": text}]}}
    return {"jsonrpc": "2.0", "id": request_id, "error": {"code": -32601, "message": "Method not found"}}
```

For an AgentCore Gateway, `connector_id` is the gateway ARN
(`arn:aws:bedrock-agentcore:<region>:<account>:gateway/<id>`), with an optional
`server_description`. What you see when the request is wrong, in September 2026:

- no `connector_id`: 400 `MCP tool 'orders' requires 'connector_id'. Note: 'server_url'
  is not supported.`
- a remote `server_url` in its place: 400 `MCP tool 'orders': 'server_url' is not
  supported. Use 'connector_id' instead.`
- a connector Bedrock cannot reach: the request fails, with `status: failed` and no
  output on gpt-oss-120b
- the same request on `bedrock-runtime`: 400 `unsupported connector_id`

gpt-oss also has built-in `notes` and `tasks` tools that need no declaration. In
September 2026 runs gpt-oss-120b called the notes tool in two of four requests, under
two different tool names and with no output, so do not build on them, and treat an
`mcp_call` item as the only sign that a tool ran.

## 2. Batch inference

The OpenAI Batch API is on `bedrock-runtime`, under `/openai/v1/batches`;
`bedrock-mantle` returns 404 on both of its paths. A job reads a JSONL file of Chat
Completions requests from S3 and writes the results beside it, in a folder named after
the batch ID. The request needs two Bedrock headers: `X-Amzn-Bedrock-RoleArn`, a service
role that can read and write the S3 location and invoke the model, and
`X-Amzn-Bedrock-ModelId`, the model's `bedrock-runtime` ID. `endpoint` must be
`/v1/chat/completions`. The batch support table lists the gpt-oss and gpt-oss-safeguard
models, not the hosted GPT models (September 2026).

```python
job = runtime.batches.create(
    input_file_id="s3://YOUR_BUCKET/batch-input.jsonl",  # an S3 URI, not a file ID
    endpoint="/v1/chat/completions",
    completion_window="24h",
    extra_headers={
        "X-Amzn-Bedrock-RoleArn": "arn:aws:iam::123456789012:role/BatchServiceRole",
        "X-Amzn-Bedrock-ModelId": "openai.gpt-oss-20b-1:0",  # the bedrock-runtime ID
    },
)
```

The cell lists the account's batch jobs, which changes nothing:

In [2]:
batches = runtime.batches.list(limit=20)
print(f"{len(batches.data)} batch job(s) in {REGION}")
for job in batches.data:
    print(f"  {job.status:12} created {job.created_at}")

0 batch job(s) in us-east-1


## 3. Reinforcement fine-tuning

Reinforcement fine-tuning trains a model from prompts and a reward function that scores
its answers. It runs through the OpenAI Files and fine-tuning jobs APIs on
`bedrock-mantle` in `us-west-2`, under the bare `/v1` path (`/openai/v1/files` returns
404), for gpt-oss-20b and Qwen3 32B. The first cell lists what the account already has
and uploads nothing.

Each line of the training file is one prompt in Chat Completions format, with a
`reference_answer` for the reward function. The service validates the file and needs 100
to 20,000 lines; the second cell builds two lines and checks their format locally. The
reward function is a Lambda that receives a list of samples, each holding the
conversation with the model's answer as the last message and the `reference_answer`
under `metadata`, and returns an `aggregate_reward_score` for each. The third cell tests
one locally on an exact answer, an answer that contains the reference, and a wrong one.

The job itself, once the file is uploaded and the reward Lambda deployed. The finished
model's ID then works on the Responses and Chat Completions APIs, with no deployment
step:

```python
with open("train.jsonl", "rb") as handle:
    uploaded = tuning.files.create(file=handle, purpose="fine-tune")

job = tuning.fine_tuning.jobs.create(
    model="openai.gpt-oss-20b",
    training_file=uploaded.id,
    extra_body={"method": {
        "type": "reinforcement",
        "reinforcement": {
            "grader": {"type": "lambda", "lambda": {"function": "arn:aws:lambda:us-west-2:123456789012:function:my-reward"}},
            "hyperparameters": {"n_epochs": 1, "batch_size": 4, "learning_rate_multiplier": 1.0},
        },
    }},
)
```

In [3]:
files = tuning.files.list()
jobs = tuning.fine_tuning.jobs.list(limit=20)
print(f"{len(files.data)} file(s) and {len(jobs.data)} fine-tuning job(s) in {FT_REGION}")
for job in jobs.data:
    print(f"  {job.status:12} {job.model}")

0 file(s) and 0 fine-tuning job(s) in us-west-2


In [4]:
ROWS = [
    {"messages": [{"role": "user", "content": "What is 12 * 12?"}], "reference_answer": "144"},
    {"messages": [{"role": "user", "content": "What is the capital of Australia?"}], "reference_answer": "Canberra"},
]
ROLES = {"system", "user", "assistant"}


def problems(line: str) -> list[str]:
    """What is wrong with one line of a training file, if anything."""
    try:
        row = json.loads(line)
    except json.JSONDecodeError as exc:
        return [f"not JSON ({exc.msg})"]
    found = []
    messages = row.get("messages")
    if not isinstance(messages, list) or not messages:
        found.append("messages must be a non-empty list")
    elif any(m.get("role") not in ROLES or not m.get("content") for m in messages):
        found.append("every message needs a known role and some content")
    if "reference_answer" not in row:
        found.append("reference_answer is missing")
    return found


jsonl = "\n".join(json.dumps(row) for row in ROWS)
bad = {n: p for n, line in enumerate(jsonl.splitlines(), 1) if (p := problems(line))}
print(jsonl)
print(f"\n{len(ROWS)} line(s); format problems: {bad or 'none'}")
print("long enough to upload" if len(ROWS) >= 100 else "too short to upload: the service needs at least 100 lines")

{"messages": [{"role": "user", "content": "What is 12 * 12?"}], "reference_answer": "144"}
{"messages": [{"role": "user", "content": "What is the capital of Australia?"}], "reference_answer": "Canberra"}

2 line(s); format problems: none
too short to upload: the service needs at least 100 lines


In [5]:
import re


def lambda_handler(event, context):
    """Score 1.0 for the exact reference answer, 0.5 if the answer names it as a whole word
    and offers no other number, else 0.0. A hedge such as "124 or 144" earns nothing, so
    training cannot learn to list candidates."""
    results = []
    for sample in event:
        answer = (sample["messages"][-1].get("content") or "").strip().lower()
        expected = str(sample.get("metadata", {}).get("reference_answer", "")).strip().lower()
        named = bool(expected) and re.search(rf"\b{re.escape(expected)}\b", answer) is not None
        other_numbers = set(re.findall(r"\d+(?:\.\d+)?", answer)) - {expected}
        score = 1.0 if answer == expected else 0.5 if named and not other_numbers else 0.0
        results.append({
            "id": sample["id"],
            "aggregate_reward_score": score,
            "metrics_list": [{"name": "exact_match", "value": float(answer == expected), "type": "Metric"}],
        })
    return results


def sample(sample_id: str, question: str, answer: str, reference: str) -> dict:
    messages = [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
    return {"id": sample_id, "messages": messages, "metadata": {"reference_answer": reference}}


event = [
    sample("exact", "What is 12 * 12?", "144", "144"),
    sample("contains", "What is the capital of Australia?", "It is Canberra.", "Canberra"),
    sample("wrong", "What is 12 * 12?", "124", "144"),
    sample("hedged", "What is 12 * 12?", "124 or 144", "144"),
]
for result in lambda_handler(event, None):
    print(f"{result['id']:9} score={result['aggregate_reward_score']}")

exact     score=1.0
contains  score=0.5
wrong     score=0.0
hedged    score=0.0


## Things that differ

What trips people up with these models, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Batch on `bedrock-mantle` | 404 on `/v1/batches` and `/openai/v1/batches` | Call `bedrock-runtime`. |
| Batch model ID | `X-Amzn-Bedrock-ModelId` takes the `bedrock-runtime` ID | Send the ID the batch support table lists, such as `openai.gpt-oss-20b-1:0`. |
| Fine-tuning path | 404 on `/openai/v1/files` and `/openai/v1/fine_tuning/jobs` | Use the bare `/v1` path on `bedrock-mantle` in `us-west-2`. |
| Training file size | The service validates the file and needs 100 to 20,000 lines | Check the format locally first (section 3). |
| Reward function contract | The Lambda receives a list of samples and returns one `aggregate_reward_score` for each sample `id` | Test the handler locally before you deploy it. |
| MCP connector | 400 on a `server_url` or a missing `connector_id` | Deploy the tool as a Lambda or an AgentCore Gateway, and pass its ARN as `connector_id`. |
| MCP connector on `bedrock-runtime` | 400 `unsupported connector_id` | Call server-side tools on `bedrock-mantle`. |
| Built-in `notes` and `tasks` tools | An `mcp_call` in some requests and not others | Do not rely on them. An `mcp_call` item is the only evidence that a tool ran. |

## Next

- Web search, the server-side tool for the hosted GPT models:
  [`02-web-search-and-grounding`](02-web-search-and-grounding.ipynb)
- Client-side function tools:
  [`03-tools-and-structured-output`](03-tools-and-structured-output.ipynb)
- Claude's agentic tools:
  [`02-anthropic-claude/03`](../02-anthropic-claude/03-agentic-computer-use-and-memory.ipynb)
- The open-weight gpt-oss models:
  [`14-openai-gpt-oss/01`](../14-openai-gpt-oss/01-gpt-oss-on-both-endpoints.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)